In [1]:
# ==========================================
# FINAL MODEL EVALUATION
# Motor Cooling ANN Project
# ==========================================

import os
import time
import ast
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neural_network import MLPRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [8]:
# ============================================================
# 03_MODEL_EVALUATION
# CELL 7: IDENTIFY FEATURES AND TARGETS
# ============================================================

import re
from difflib import get_close_matches

# Normalize column names to handle spaces, symbols and units
def normalize_column(name):
    name = str(name).lower().strip()
    name = name.replace("°", "")
    return re.sub(r"[^a-z0-9]", "", name)


# Create a lookup of the actual dataset columns
column_lookup = {
    normalize_column(col): col
    for col in df.columns
}


# Find a column using multiple possible names
def find_column(aliases, description):
    # Exact normalized match
    for alias in aliases:
        key = normalize_column(alias)
        if key in column_lookup:
            return column_lookup[key]

    # Match common variations of column names
    available = list(column_lookup.keys())

    for alias in aliases:
        key = normalize_column(alias)

        matches = [
            col for col in available
            if key in col or col in key
        ]

        if len(matches) == 1:
            return column_lookup[matches[0]]

    # Try a close match if necessary
    for alias in aliases:
        key = normalize_column(alias)
        matches = get_close_matches(
            key, available, n=1, cutoff=0.75
        )

        if matches:
            return column_lookup[matches[0]]

    raise ValueError(
        f"Cannot identify {description}.\n"
        f"Available columns: {df.columns.tolist()}"
    )


# 1. Identify input features
feature_aliases = {
    "Coolant Flow Rate": [
        "Coolant Flow Rate (LPH)",
        "Coolant Flow Rate"
    ],
    "Nanoparticle Concentration": [
        "Nanoparticle Concentration (%)",
        "Nanoparticle Concentration"
    ],
    "Motor Speed": [
        "Motor Speed (rpm)",
        "Motor Speed"
    ],
    "Motor Load": [
        "Motor Load (%)",
        "Motor Load"
    ]
}

feature_names = [
    find_column(aliases, name)
    for name, aliases in feature_aliases.items()
]


# 2. Identify output targets
target_aliases = {
    "Winding Temperature": [
        "Winding Temperature (°C)",
        "Winding Temperature",
        "Winding Temp"
    ],
    "Motor Efficiency": [
        "Motor Efficiency (%)",
        "Motor Efficiency"
    ],
    "Cooling Efficiency": [
        "Cooling Efficiency (%)",
        "Cooling Efficiency"
    ],
    "Case Temperature": [
        "Case Temperature (°C)",
        "Case Temperature (C)",
        "Case Temperature",
        "Case Temp (°C)",
        "Case Temp (C)",
        "Case Temp"
    ],
    "Pumping Power": [
        "Pumping Power (W)",
        "Pumping Power",
        "Pump Power (W)",
        "Pump Power"
    ]
}

target_names = [
    find_column(aliases, name)
    for name, aliases in target_aliases.items()
]


# 3. Validate the selected columns
print("=" * 60)
print("FEATURE AND TARGET IDENTIFICATION")
print("=" * 60)

print("\nInput features:")
for col in feature_names:
    print("  ", col)

print("\nTarget variables:")
for col in target_names:
    print("  ", col)


# 4. Prepare the input and output data
for col in feature_names + target_names:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(
    subset=feature_names + target_names
).reset_index(drop=True)

X = df[feature_names].copy()
Y = df[target_names].copy()

print("\nDataset prepared successfully!")
print("Dataset shape:", df.shape)
print("Input shape:", X.shape)
print("Target shape:", Y.shape)

print("\nAll five target variables identified successfully!")

FEATURE AND TARGET IDENTIFICATION

Input features:
   Coolant Flow Rate (LPH)
   Nanoparticle Concentration (%)
   Motor Speed (rpm)
   Motor Load (%)

Target variables:
   Winding Temperature (°C)
   Motor Efficiency (%)
   Cooling Efficiency (%)
   Casing Temp (°C)
   Pumping Power (W)

Dataset prepared successfully!
Dataset shape: (108, 10)
Input shape: (108, 4)
Target shape: (108, 5)

All five target variables identified successfully!


In [9]:
# ============================================================
# CELL 3: FIND SAVED ANN MODELS
# ============================================================

from pathlib import Path

# Find the project directory
project_dir = Path.cwd()

while not (project_dir / "data").exists():
    if project_dir.parent == project_dir:
        raise FileNotFoundError("Project directory not found.")
    project_dir = project_dir.parent

# Locate the models folder
models_dir = project_dir / "models"

print("=" * 60)
print("SAVED ANN MODELS")
print("=" * 60)

if not models_dir.exists():
    print("Models folder does not exist.")
else:
    model_files = [
        p for p in models_dir.rglob("*")
        if p.is_file()
    ]

    if not model_files:
        print("No saved model files found.")
    else:
        for file in model_files:
            print(f"\nFile: {file.relative_to(project_dir)}")
            print(f"Size: {file.stat().st_size / 1024:.2f} KB")

# Check available experiment results
results_dir = project_dir / "experiments" / "results"

print("\n" + "=" * 60)
print("EXPERIMENT RESULTS")
print("=" * 60)

if results_dir.exists():
    for file in sorted(results_dir.glob("*.csv")):
        print("-", file.name)
else:
    print("Experiment results folder not found.")

SAVED ANN MODELS
No saved model files found.

EXPERIMENT RESULTS
- activation_results.csv
- algorithm_results.csv
- baseline_results.csv
- epoch_results.csv
- epochs_results.csv
- hidden_layer_results.csv
- neuron_results.csv
- training_algorithm_results.csv
